# MVP — Da base bruta ao plano de retenção: um pipeline de dados em nuvem

**Universidade de Brasília — Departamento de Engenharia de Produção**
Projeto de Sistema de Produção 8 · Prof. André Luiz Marques Serrano
Isadora Messenberg Guimarães Macêdo Rebello — matrícula 231013387

---

## 1. Objetivo

Uma operadora de telecomunicações perde **26,5% da sua base** por ciclo. Hoje a área
comercial reage ao cancelamento depois que ele acontece: quando o cliente liga para
cancelar, a retenção já custa caro e funciona pouco.

Este MVP testa uma hipótese: **é possível, a partir apenas do cadastro e da fatura,
montar um pipeline que entregue semanalmente ao time comercial uma lista priorizada
de clientes em risco, com a ação de retenção indicada para cada um e o retorno
esperado dessa ação.**

O produto mínimo para testar essa hipótese é o pipeline completo — da ingestão do
arquivo bruto até o plano de ação persistido na nuvem.

### Perguntas de negócio

| # | Pergunta | Decisão que ela sustenta |
|---|---|---|
| **P1** | Quanto da receita mensal está em risco e em quais combinações de contrato, tecnologia e forma de pagamento ela se concentra? | Onde focar a oferta comercial |
| **P2** | Quais atributos explicam o cancelamento **depois de controlar os demais**, e em que isso difere da leitura uma-variável-por-vez? | Quais alavancas de produto priorizar |
| **P3** | É possível identificar quem vai cancelar antes do pedido de cancelamento, e esse ganho sobrevive a amostras diferentes? | Se vale operar um modelo |
| **P4** | As probabilidades do modelo são confiáveis o bastante para servirem de base a contas em dinheiro? | Se o valor esperado calculado é crível |
| **P5** | Com orçamento limitado, qual lista e qual ação maximizam o retorno — e o ganho sobre uma **regra de negócio bem construída** é real? | Quanto investir e em quem |
| **P6** | O modelo trata grupos demográficos de forma equilibrada? | Risco reputacional e de conformidade (LGPD) |

> As perguntas permanecem íntegras até o fim do documento, inclusive as respondidas
> de forma negativa. A seção 9 (autoavaliação) discute o atingimento de cada uma.

---

## 2. Arquitetura

```
  Kaggle (CSV)
       │  download manual, upload no Volume
       ▼
  /Volumes/workspace/default/churn/
       │
       ▼
  ┌─────────────┐   ingestão fiel, tudo texto, sem transformação
  │   BRONZE    │   bronze_telco_raw          + metadados de linhagem
  └─────────────┘
       │  perfilamento de qualidade (6 dimensões)
       ├──────────────────► qa_qualidade_atributos · qa_regras_consistencia
       ▼
  ┌─────────────┐   9 regras de transformação numeradas e documentadas
  │   SILVER    │   silver_assinatura
  └─────────────┘
       │
       ▼
  ┌─────────────┐   esquema estrela: 1 fato + 6 dimensões
  │    GOLD     │   fato_assinatura · dim_cliente · dim_demografia
  │             │   dim_contrato · dim_pagamento · dim_internet · dim_telefonia
  └─────────────┘   + meta_catalogo_dados (catálogo com domínio e linhagem)
       │
       ▼
  análise descritiva · diagnóstica · preditiva · prescritiva
       │
       ▼
  plano_de_acao_retencao.csv  (Volume)  +  gold_plano_retencao (tabela Delta)
```

Todas as tabelas são **Delta**, persistidas no Unity Catalog em
`workspace.churn_mvp`. Nada depende da sessão: depois que o notebook roda, as
tabelas continuam lá e podem ser consultadas por SQL.

---

## 3. Coleta

| Item | Registro |
|---|---|
| Fonte | Kaggle — `blastchar/telco-customer-churn` (IBM Sample Data Sets) |
| URL | https://www.kaggle.com/datasets/blastchar/telco-customer-churn |
| Licença | Database Contents License (DbCL) v1.0 — permite uso e redistribuição com atribuição |
| Formato original | CSV único, UTF-8, separador vírgula, cabeçalho na 1ª linha |
| Volume | 7.043 registros × 21 atributos · 970.457 bytes |
| Técnica | download manual e upload para o Volume (a API do Kaggle exige credencial pessoal, que não deve ser versionada) |
| Dados pessoais | não há identificadores diretos. O `customerID` é um código sintético sem vínculo com pessoa natural; a base é pública e anonimizada na origem. Não se aplica tratamento adicional de anonimização sob a Lei 13.709/2018 |

> **Antes de rodar:** baixe o CSV do Kaggle e envie para o Volume.
> *Catalog → workspace → default → Create → Volume* com o nome `churn`, depois
> *Upload to this volume*. Mantenha o nome `WA_Fn-UseC_-Telco-Customer-Churn.csv`.

In [ ]:
# Bibliotecas que não vêm no runtime do Databricks
%pip install -q scikit-learn==1.5.2
dbutils.library.restartPython()

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window

CATALOGO, SCHEMA = "workspace", "churn_mvp"
VOLUME  = "/Volumes/workspace/default/churn"
ARQUIVO = f"{VOLUME}/WA_Fn-UseC_-Telco-Customer-Churn.csv"
FQ = lambda t: f"{CATALOGO}.{SCHEMA}.{t}"          # nome qualificado da tabela

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA} "
          f"COMMENT 'MVP PSP8 — pipeline de retenção de clientes'")
print("schema pronto:", f"{CATALOGO}.{SCHEMA}")

# Confere que o arquivo está no Volume antes de qualquer coisa
try:
    dbutils.fs.ls(ARQUIVO)
    print("arquivo encontrado:", ARQUIVO)
except Exception:
    raise FileNotFoundError(
        f"CSV não encontrado em {ARQUIVO}. Suba o arquivo no Volume 'churn' "
        f"(Catalog > workspace > default > churn > Upload to this volume).")

---
## 4. Camada Bronze — ingestão fiel

A bronze guarda o dado **exatamente como chegou**: todas as colunas como texto,
nenhuma conversão, nenhuma limpeza. Isso é deliberado — se uma regra de
transformação estiver errada, dá para reprocessar a partir da bronze sem voltar
ao Kaggle. Acrescentamos apenas metadados de linhagem.

In [ ]:
bronze = (spark.read
          .option("header", True)
          .option("inferSchema", False)        # tudo texto: a bronze não interpreta
          .csv(ARQUIVO)
          .withColumn("_ingerido_em",    F.current_timestamp())
          .withColumn("_arquivo_origem", F.col("_metadata.file_path"))   # input_file_name() não existe no Unity Catalog
          .withColumn("_camada",         F.lit("bronze")))

(bronze.write.mode("overwrite").option("overwriteSchema", True)
       .saveAsTable(FQ("bronze_telco_raw")))

b = spark.table(FQ("bronze_telco_raw"))
N = b.count()
print(f"bronze_telco_raw: {N} linhas x {len(b.columns)} colunas")
display(b.limit(5))

---
## 5. Análise de qualidade dos dados

Avaliamos as seis dimensões usuais, **atributo a atributo**, e persistimos o
resultado como tabela — o perfil de qualidade é parte do produto, não um print
perdido no meio do notebook.

| Dimensão | O que medimos |
|---|---|
| Completude | vazios e nulos por atributo |
| Unicidade | duplicatas de chave e linhas integralmente repetidas |
| Consistência | regras cruzadas entre atributos (R1 a R6) |
| Conformidade | aderência ao domínio declarado |
| Acurácia | valores implausíveis e coerência aritmética |
| Atualidade | existência de marcação temporal |

Primeiro declaramos o **domínio esperado** de cada atributo. Sem domínio declarado
não existe conformidade: só se pode dizer que um valor é inválido contra um
contrato explícito.

In [ ]:
SIM_NAO   = ["Yes", "No"]
SERV_NET  = ["Yes", "No", "No internet service"]

DOMINIOS = {
  "customerID":      ("texto",      None),
  "gender":          ("categórico", ["Female", "Male"]),
  "SeniorCitizen":   ("binário",    ["0", "1"]),
  "Partner":         ("categórico", SIM_NAO),
  "Dependents":      ("categórico", SIM_NAO),
  "tenure":          ("numérico",   (0, 72)),
  "PhoneService":    ("categórico", SIM_NAO),
  "MultipleLines":   ("categórico", ["Yes", "No", "No phone service"]),
  "InternetService": ("categórico", ["DSL", "Fiber optic", "No"]),
  "OnlineSecurity":  ("categórico", SERV_NET),
  "OnlineBackup":    ("categórico", SERV_NET),
  "DeviceProtection":("categórico", SERV_NET),
  "TechSupport":     ("categórico", SERV_NET),
  "StreamingTV":     ("categórico", SERV_NET),
  "StreamingMovies": ("categórico", SERV_NET),
  "Contract":        ("categórico", ["Month-to-month", "One year", "Two year"]),
  "PaperlessBilling":("categórico", SIM_NAO),
  "PaymentMethod":   ("categórico", ["Electronic check", "Mailed check",
                                     "Bank transfer (automatic)", "Credit card (automatic)"]),
  "MonthlyCharges":  ("numérico",   (18.0, 120.0)),
  "TotalCharges":    ("numérico",   (0.0, 9000.0)),
  "Churn":           ("binário",    SIM_NAO),
}

def num(coluna):
    """Texto -> número tolerando vazios. O Spark opera em modo ANSI: um cast
    direto de '' para DOUBLE aborta o job, por isso try_cast + nullif."""
    return F.expr(f"try_cast(nullif(trim(`{coluna}`), '') AS DOUBLE)")

In [ ]:
linhas = []
for col, (tipo, dom) in DOMINIOS.items():
    c = F.trim(F.col(col))
    vazios    = b.filter(c.isNull() | (c == "")).count()
    distintos = b.select(col).distinct().count()

    if isinstance(dom, list):                      # domínio categórico
        fora, dom_txt = b.filter((c != "") & ~c.isin(dom)).count(), " | ".join(dom)
    elif isinstance(dom, tuple):                   # faixa numérica
        v = num(col)
        fora = b.filter(v.isNotNull() & ((v < dom[0]) | (v > dom[1]))).count()
        dom_txt = f"[{dom[0]}, {dom[1]}]"
    else:                                          # padrão textual
        fora, dom_txt = b.filter(~c.rlike(r"^\d{4}-[A-Z]{5}$")).count(), "padrão ####-XXXXX"

    linhas.append((col, tipo, dom_txt, distintos, vazios,
                   1 - vazios/N, fora, 1 - fora/N))

esquema = T.StructType([
    T.StructField("atributo", T.StringType()),  T.StructField("tipo", T.StringType()),
    T.StructField("dominio", T.StringType()),   T.StructField("valores_distintos", T.LongType()),
    T.StructField("vazios", T.LongType()),      T.StructField("completude", T.DoubleType()),
    T.StructField("fora_dominio", T.LongType()),T.StructField("conformidade", T.DoubleType())])

qa = (spark.createDataFrame(linhas, esquema)
           .withColumn("_avaliado_em", F.current_timestamp()))
(qa.write.mode("overwrite").option("overwriteSchema", True)
   .saveAsTable(FQ("qa_qualidade_atributos")))

print("atributos com alguma falha de completude ou conformidade:")
display(qa.filter("completude < 1 OR conformidade < 1"))

**Completude e conformidade.** Um único atributo falha: `TotalCharges`, com 11
vazios (completude 99,84%). Nenhum valor fora do domínio declarado em nenhum
atributo — a base chega curada da origem, o que o enunciado prevê como caso
possível. Note que *demonstrar* a ausência de problema exige a mesma verificação
que encontrá-lo.

Agora as regras cruzadas, que é onde a base realmente é testada.

In [ ]:
NET = ["OnlineSecurity","OnlineBackup","DeviceProtection",
       "TechSupport","StreamingTV","StreamingMovies"]

# R1: quem não tem internet precisa estar marcado como "No internet service"
# em TODOS os serviços agregados — e quem tem, em nenhum.
cond_r1 = None
for c in NET:
    x = (((F.col("InternetService") == "No") & (F.col(c) != "No internet service")) |
         ((F.col("InternetService") != "No") & (F.col(c) == "No internet service")))
    cond_r1 = x if cond_r1 is None else (cond_r1 | x)

tc, tn, mc = num("TotalCharges"), num("tenure"), num("MonthlyCharges")

regras = [
 ("R1", 'InternetService = "No" <=> serviços agregados = "No internet service"',
  b.filter(cond_r1).count()),
 ("R2", 'PhoneService = "No" <=> MultipleLines = "No phone service"',
  b.filter(((F.col("PhoneService") == "No") & (F.col("MultipleLines") != "No phone service")) |
           ((F.col("PhoneService") != "No") & (F.col("MultipleLines") == "No phone service"))).count()),
 ("R3", "tenure = 0 <=> TotalCharges vazio",
  b.filter(((tn == 0) & tc.isNotNull()) | ((tn > 0) & tc.isNull())).count()),
 ("R4", "TotalCharges ~ tenure x MonthlyCharges (tolerância 20%)",
  b.filter((tn > 0) & tc.isNotNull() & (F.abs(tc - tn*mc)/(tn*mc) > 0.20)).count()),
 ("R5", "customerID é único",
  N - b.select("customerID").distinct().count()),
 ("R6", "linhas integralmente duplicadas (desconsiderando o ID)",
  N - b.drop("customerID","_ingerido_em","_arquivo_origem","_camada").distinct().count()),
]

rq = (spark.createDataFrame(regras, ["regra","descricao","violacoes"])
           .withColumn("situacao", F.when(F.col("violacoes") == 0, "aprovado").otherwise("atenção"))
           .withColumn("_avaliado_em", F.current_timestamp()))
(rq.write.mode("overwrite").option("overwriteSchema", True)
   .saveAsTable(FQ("qa_regras_consistencia")))
display(rq)

In [ ]:
# Acurácia: quão longe TotalCharges fica do que a aritmética prevê
erro = (b.filter((tn > 0) & tc.isNotNull())
         .select((F.abs(tc - tn*mc)/(tn*mc)).alias("erro_relativo")))
display(erro.selectExpr("percentile(erro_relativo, 0.50) AS mediana",
                        "percentile(erro_relativo, 0.95) AS p95",
                        "percentile(erro_relativo, 0.99) AS p99",
                        "max(erro_relativo) AS maximo"))

### O que a análise de qualidade encontrou

**R4 — 59 registros (0,8%)** têm `TotalCharges` divergindo mais de 20% do produto
`tenure × MonthlyCharges`. O erro mediano é de 2%. A leitura correta não é "erro de
dado": é que `MonthlyCharges` registra a **tarifa vigente**, não a média histórica.
Quem teve reajuste ou trocou de plano aparece como divergência. Isso tem uma
consequência de modelagem: `MonthlyCharges` e `TotalCharges` medem momentos
diferentes e não podem ser tratados como redundantes. A regra T7 adiante cria o
`ticket_medio_historico` justamente para separar os dois conceitos.

**R6 — 22 linhas (0,31%)** são idênticas em todos os 20 atributos, diferindo apenas
pelo `customerID`. Não são duplicatas de carga: são clientes distintos com perfil
igual, plausível numa base de 7 mil registros e 20 atributos majoritariamente
binários. **Mantivemos as 22 linhas.** Removê-las descartaria clientes reais.

**R1, R2, R3, R5 — aprovados sem exceção.** A base é internamente coerente.

**Atualidade — a falha estrutural.** A base **não possui nenhum carimbo de tempo**:
não há data de referência, de adesão ou de cancelamento. É um retrato sem data.
Três consequências, que condicionam tudo o que vem depois:

1. Não há como saber a que período o `Churn` se refere, nem o tamanho da janela de observação.
2. Não é possível validação temporal (treinar no passado, testar no futuro), que é o protocolo correto para previsão de churn. Somos obrigados a usar divisão aleatória, que assume que o tempo não importa.
3. Não dá para reconstruir a trajetória do cliente, só o estado final.

Esta limitação é registrada aqui e retomada na autoavaliação.

---
## 6. Camada Silver — transformação

Nove regras de negócio, cada uma numerada, justificada e com o volume afetado.
Elas afetam diretamente a interpretação dos resultados, então ficam explícitas.

| ID | Origem | Regra | Justificativa |
|---|---|---|---|
| T1 | `TotalCharges` | texto → numérico; vazio recebe `0,00` | pela R3, todo vazio é cliente com `tenure=0`: ainda não houve faturamento, então zero é o valor correto, não um dado faltante |
| T2 | `tenure`, `MonthlyCharges` | texto → numérico | a bronze guarda tudo como texto |
| T3 | `SeniorCitizen` | `0/1` → `No/Yes` | alinha com as demais flags; evita que o modelo trate uma categoria como grandeza ordinal |
| T4 | 7 atributos de serviço | `"No internet service"` → `"No"` | a ausência do serviço-pai já está em `InternetService`. Manter o terceiro nível duplica a informação em 7 colunas e infla a dimensionalidade sem acrescentar nada |
| T5 | `Churn` | `Yes/No` → `1/0` | variável-alvo numérica |
| T6 | `PaymentMethod` | deriva `pagamento_automatico` | separa o *instrumento* (cartão, boleto) do *mecanismo* (automático ou manual), que é a alavanca acionável |
| T7 | `TotalCharges`, `tenure` | deriva `ticket_medio_historico` | tarifa média efetivamente paga, distinta da vigente |
| T8 | T7, `MonthlyCharges` | deriva `desvio_tarifario` | hipótese: reajuste recente antecede cancelamento |
| T9 | 6 atributos de serviço | deriva `n_servicos` | profundidade de relacionamento num único número |

> **T8 é uma hipótese que será testada e rejeitada.** Veja a seção 8.2: o atributo
> fica em 22º lugar entre 23 em importância. Mantemos o atributo e o registro do
> resultado negativo, em vez de apagar a tentativa.

In [ ]:
s = (b
     .withColumn("tenure_meses",    tn.cast("int"))                                    # T2
     .withColumn("valor_mensal",    mc)                                                # T2
     .withColumn("valor_acumulado", F.coalesce(tc, F.lit(0.0)))                        # T1
     .withColumn("idoso",      F.when(F.col("SeniorCitizen") == "1", "Yes").otherwise("No"))   # T3
     .withColumn("flag_churn", F.when(F.col("Churn") == "Yes", 1).otherwise(0))                # T5
     .withColumn("pagamento_automatico",
                 F.when(F.col("PaymentMethod").contains("automatic"), "Yes").otherwise("No"))) # T6

for c in NET:                                                                           # T4
    s = s.withColumn(c, F.when(F.col(c) == "No internet service", "No").otherwise(F.col(c)))
s = s.withColumn("MultipleLines",
                 F.when(F.col("MultipleLines") == "No phone service", "No").otherwise(F.col("MultipleLines")))

s = (s
     .withColumn("ticket_medio_historico",                                              # T7
                 F.when(F.col("tenure_meses") > 0,
                        F.col("valor_acumulado")/F.col("tenure_meses"))
                  .otherwise(F.col("valor_mensal")))
     .withColumn("desvio_tarifario",                                                    # T8
                 F.col("valor_mensal") - F.col("ticket_medio_historico"))
     .withColumn("n_servicos",                                                          # T9
                 sum([F.when(F.col(c) == "Yes", 1).otherwise(0) for c in NET]))
     .withColumn("_camada", F.lit("silver"))
     .drop("SeniorCitizen", "Churn", "tenure", "MonthlyCharges", "TotalCharges"))

(s.write.mode("overwrite").option("overwriteSchema", True)
   .saveAsTable(FQ("silver_assinatura")))

sv = spark.table(FQ("silver_assinatura"))
assert sv.count() == N, "a transformação alterou o número de linhas"
print(f"silver_assinatura: {sv.count()} linhas (nenhuma perda)")
display(sv.select("customerID","tenure_meses","valor_mensal","valor_acumulado",
                  "ticket_medio_historico","desvio_tarifario","n_servicos","flag_churn").limit(5))

---
## 7. Camada Gold — esquema estrela e catálogo

### 7.1 Por que estrela, e com quais dimensões

O grão do fato é **um cliente, um retrato**. Cada dimensão agrupa atributos que
descrevem o mesmo conceito de negócio e mudam juntos:

| Dimensão | Conceito | Atributos |
|---|---|---|
| `dim_cliente` | identidade | `customerID` |
| `dim_demografia` | perfil do titular | gênero, idoso, cônjuge, dependentes |
| `dim_contrato` | vínculo comercial | modalidade, fatura digital |
| `dim_pagamento` | instrumento financeiro | método, automático |
| `dim_internet` | pacote de banda larga | tecnologia + 6 serviços agregados |
| `dim_telefonia` | pacote de voz | telefonia, múltiplas linhas |

`dim_internet` é uma **dimensão-lixo** (*junk dimension*): sete atributos de baixa
cardinalidade que só fazem sentido juntos. Em vez de sete colunas no fato, uma única
chave resolve as 129 combinações que de fato ocorrem — de 2.187 teoricamente possíveis.
Essa é a economia que o modelo dimensional entrega: o fato fica estreito e as consultas
de negócio ficam legíveis.

As métricas (`tenure_meses`, `valor_mensal`, `valor_acumulado` e as derivadas) ficam no
fato por serem aditivas ou semi-aditivas ao longo das dimensões.

In [ ]:
DIMENSOES = {
 "dim_demografia": (["gender", "idoso", "Partner", "Dependents"],      "sk_demografia"),
 "dim_contrato":   (["Contract", "PaperlessBilling"],                  "sk_contrato"),
 "dim_pagamento":  (["PaymentMethod", "pagamento_automatico"],         "sk_pagamento"),
 "dim_internet":   (["InternetService"] + NET,                         "sk_internet"),
 "dim_telefonia":  (["PhoneService", "MultipleLines"],                 "sk_telefonia"),
}

fato = sv
for nome, (cols, sk) in DIMENSOES.items():
    d = (sv.select(*cols).distinct()
           .withColumn(sk, F.row_number().over(Window.orderBy(*cols))))   # chave substituta
    (d.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(FQ(nome)))
    fato = fato.join(spark.table(FQ(nome)), on=cols, how="inner")
    print(f"{nome:16s} {spark.table(FQ(nome)).count():4d} combinações distintas")

dim_cli = (sv.select("customerID").distinct()
             .withColumn("sk_cliente", F.row_number().over(Window.orderBy("customerID"))))
(dim_cli.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(FQ("dim_cliente")))

fato = (fato.join(spark.table(FQ("dim_cliente")), on="customerID")
            .select("sk_cliente","sk_demografia","sk_contrato","sk_pagamento",
                    "sk_internet","sk_telefonia","tenure_meses","valor_mensal",
                    "valor_acumulado","ticket_medio_historico","desvio_tarifario",
                    "n_servicos","flag_churn")
            .withColumn("_camada", F.lit("gold")))
(fato.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(FQ("fato_assinatura")))

In [ ]:
# Validação da carga: o teste que diz se o ETL está correto
f = spark.table(FQ("fato_assinatura"))
n_fato   = f.count()
n_chaves = f.select("sk_cliente").distinct().count()
n_orfaos = (f.join(spark.table(FQ("dim_internet")), "sk_internet", "left_anti").count() +
            f.join(spark.table(FQ("dim_contrato")), "sk_contrato", "left_anti").count() +
            f.join(spark.table(FQ("dim_pagamento")), "sk_pagamento", "left_anti").count())

print(f"linhas no fato        : {n_fato}  (bronze: {N})")
print(f"chaves distintas      : {n_chaves}")
print(f"chaves órfãs nas dims : {n_orfaos}")
assert n_fato == N,      "FALHA: a carga perdeu ou duplicou linhas"
assert n_chaves == N,    "FALHA: o grão de 1 linha por cliente foi violado"
assert n_orfaos == 0,    "FALHA: há chave estrangeira sem correspondência"
print("\nvalidação da carga: APROVADA — sem perda, sem duplicação, sem órfãos")

### 7.2 Catálogo de dados

O catálogo é uma **tabela consultável**, não um anexo. Quem for usar o modelo depois
descobre tipo, significado, domínio, obrigatoriedade e origem de cada atributo com
uma consulta SQL.

In [ ]:
CATALOGO_DADOS = [
 # tabela, atributo, tipo, descrição, domínio, obrigatório, linhagem
 ("fato_assinatura","sk_cliente","inteiro","chave substituta do cliente","1..7043","sim","dim_cliente.sk_cliente"),
 ("fato_assinatura","sk_demografia","inteiro","FK do perfil demográfico","1..16","sim","dim_demografia.sk_demografia"),
 ("fato_assinatura","sk_contrato","inteiro","FK do vínculo comercial","1..6","sim","dim_contrato.sk_contrato"),
 ("fato_assinatura","sk_pagamento","inteiro","FK do instrumento de pagamento","1..4","sim","dim_pagamento.sk_pagamento"),
 ("fato_assinatura","sk_internet","inteiro","FK do pacote de banda larga","1..129","sim","dim_internet.sk_internet"),
 ("fato_assinatura","sk_telefonia","inteiro","FK do pacote de voz","1..3","sim","dim_telefonia.sk_telefonia"),
 ("fato_assinatura","tenure_meses","inteiro","meses decorridos desde a adesão","[0, 72]","sim","bronze.tenure via T2"),
 ("fato_assinatura","valor_mensal","decimal","cobrança do ciclo corrente, US$","[18,25; 118,75]","sim","bronze.MonthlyCharges via T2"),
 ("fato_assinatura","valor_acumulado","decimal","receita acumulada desde a adesão, US$","[0; 8.684,80]","sim","bronze.TotalCharges via T1"),
 ("fato_assinatura","ticket_medio_historico","decimal","valor_acumulado / tenure_meses","[18,25; 118,75]","sim","derivado T7"),
 ("fato_assinatura","desvio_tarifario","decimal","valor_mensal − ticket_medio_historico","[−18; +20]","sim","derivado T8"),
 ("fato_assinatura","n_servicos","inteiro","serviços agregados contratados","[0, 6]","sim","derivado T9"),
 ("fato_assinatura","flag_churn","binário","1 = cancelou no ciclo observado","{0, 1}","sim","bronze.Churn via T5"),
 ("dim_cliente","customerID","texto","identificador natural do cliente","padrão ####-XXXXX","sim","bronze.customerID"),
 ("dim_demografia","gender","categórico","gênero declarado","Female | Male","sim","bronze.gender"),
 ("dim_demografia","idoso","categórico","65 anos ou mais","Yes | No","sim","bronze.SeniorCitizen via T3"),
 ("dim_demografia","Partner","categórico","possui cônjuge","Yes | No","sim","bronze.Partner"),
 ("dim_demografia","Dependents","categórico","possui dependentes","Yes | No","sim","bronze.Dependents"),
 ("dim_contrato","Contract","categórico","modalidade contratual","Month-to-month | One year | Two year","sim","bronze.Contract"),
 ("dim_contrato","PaperlessBilling","categórico","fatura digital","Yes | No","sim","bronze.PaperlessBilling"),
 ("dim_pagamento","PaymentMethod","categórico","instrumento de pagamento","Electronic check | Mailed check | Bank transfer (automatic) | Credit card (automatic)","sim","bronze.PaymentMethod"),
 ("dim_pagamento","pagamento_automatico","categórico","débito ou crédito automático","Yes | No","sim","derivado T6"),
 ("dim_internet","InternetService","categórico","tecnologia de acesso","DSL | Fiber optic | No","sim","bronze.InternetService"),
 ("dim_internet","OnlineSecurity","categórico","segurança online contratada","Yes | No","sim","bronze via T4"),
 ("dim_internet","OnlineBackup","categórico","backup em nuvem contratado","Yes | No","sim","bronze via T4"),
 ("dim_internet","DeviceProtection","categórico","proteção de equipamento","Yes | No","sim","bronze via T4"),
 ("dim_internet","TechSupport","categórico","suporte técnico contratado","Yes | No","sim","bronze via T4"),
 ("dim_internet","StreamingTV","categórico","streaming de TV","Yes | No","sim","bronze via T4"),
 ("dim_internet","StreamingMovies","categórico","streaming de filmes","Yes | No","sim","bronze via T4"),
 ("dim_telefonia","PhoneService","categórico","assina telefonia fixa","Yes | No","sim","bronze.PhoneService"),
 ("dim_telefonia","MultipleLines","categórico","possui múltiplas linhas","Yes | No","sim","bronze via T4"),
]

cat_df = spark.createDataFrame(
    CATALOGO_DADOS,
    ["tabela","atributo","tipo","descricao","dominio","obrigatorio","linhagem"])
(cat_df.write.mode("overwrite").option("overwriteSchema", True)
       .saveAsTable(FQ("meta_catalogo_dados")))
print(f"{cat_df.count()} atributos catalogados")
display(cat_df)

In [ ]:
# O catálogo também é aplicado como metadado nativo das tabelas Delta,
# para que apareça na interface do Unity Catalog
for tabela, atributo, _, descricao, dominio, _, linhagem in CATALOGO_DADOS:
    comentario = f"{descricao}. Domínio: {dominio}. Linhagem: {linhagem}".replace("'", "''")
    try:
        spark.sql(f"ALTER TABLE {FQ(tabela)} ALTER COLUMN `{atributo}` COMMENT '{comentario}'")
    except Exception as e:
        print(f"[aviso] {tabela}.{atributo}: {str(e)[:70]}")
print("comentários aplicados — confira em Catalog > workspace > churn_mvp")

### 7.3 Consultando o modelo estrela

Com o estrela pronto, perguntas de negócio viram SQL curto. É esse o teste de que a
modelagem valeu a pena.

In [ ]:
%sql
-- P1: onde está a receita mensal em risco
SELECT c.Contract                                   AS contrato,
       p.PaymentMethod                              AS pagamento,
       i.InternetService                            AS internet,
       COUNT(*)                                     AS clientes,
       ROUND(AVG(f.flag_churn), 3)                  AS taxa_churn,
       ROUND(SUM(CASE WHEN f.flag_churn = 1 THEN f.valor_mensal ELSE 0 END), 0) AS receita_em_risco
FROM   workspace.churn_mvp.fato_assinatura f
JOIN   workspace.churn_mvp.dim_contrato   c ON f.sk_contrato  = c.sk_contrato
JOIN   workspace.churn_mvp.dim_pagamento  p ON f.sk_pagamento = p.sk_pagamento
JOIN   workspace.churn_mvp.dim_internet   i ON f.sk_internet  = i.sk_internet
GROUP  BY c.Contract, p.PaymentMethod, i.InternetService
HAVING COUNT(*) >= 50
ORDER  BY receita_em_risco DESC
LIMIT  10

In [ ]:
%sql
-- Concentração: churn pesa mais na receita do que na contagem de clientes
SELECT ROUND(SUM(CASE WHEN flag_churn = 1 THEN valor_mensal ELSE 0 END), 0) AS receita_cancelada,
       ROUND(SUM(valor_mensal), 0)                                          AS receita_total,
       ROUND(SUM(CASE WHEN flag_churn = 1 THEN valor_mensal ELSE 0 END)
             / SUM(valor_mensal), 4)                                        AS pct_receita,
       ROUND(AVG(flag_churn), 4)                                            AS pct_clientes,
       ROUND(AVG(CASE WHEN flag_churn = 1 THEN valor_mensal END), 2)        AS ticket_cancelou,
       ROUND(AVG(CASE WHEN flag_churn = 0 THEN valor_mensal END), 2)        AS ticket_ficou
FROM   workspace.churn_mvp.fato_assinatura

**Resposta a P1.** O churn atinge **26,5% dos clientes, mas 30,5% da receita**:
US$ 139.131 de US$ 456.117 mensais. Quem cancela paga US$ 74,44 contra US$ 61,27 de
quem fica — **21% a mais**. A perda é seletiva e recai sobre o cliente mais valioso,
o que muda a conta de quanto vale investir em retenção.

A concentração é nítida: contrato mensal responde por US$ 120.847 da receita em risco,
fibra por US$ 114.300 e cheque eletrônico por US$ 84.289. São os mesmos clientes vistos
por três ângulos, e é sobre essa interseção que o plano de ação vai operar.

---
## 8. Análise — solução do problema

A partir daqui saímos do Spark e trabalhamos em pandas/scikit-learn sobre a camada
Gold. A base cabe em memória (7 mil linhas), então distribuir o treinamento só
acrescentaria complexidade sem ganho.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, brier_score_loss, silhouette_score)
from scipy.stats import chi2
import matplotlib.pyplot as plt

SEED = 42

# Reconstitui a visão analítica juntando o fato às dimensões
vis = spark.sql(f"""
  SELECT cl.customerID, dm.*, ct.*, pg.*, it.*, tl.*,
         f.tenure_meses, f.valor_mensal, f.valor_acumulado,
         f.ticket_medio_historico, f.desvio_tarifario, f.n_servicos, f.flag_churn
  FROM {FQ('fato_assinatura')} f
  JOIN {FQ('dim_cliente')}     cl ON f.sk_cliente    = cl.sk_cliente
  JOIN {FQ('dim_demografia')}  dm ON f.sk_demografia = dm.sk_demografia
  JOIN {FQ('dim_contrato')}    ct ON f.sk_contrato   = ct.sk_contrato
  JOIN {FQ('dim_pagamento')}   pg ON f.sk_pagamento  = pg.sk_pagamento
  JOIN {FQ('dim_internet')}    it ON f.sk_internet   = it.sk_internet
  JOIN {FQ('dim_telefonia')}   tl ON f.sk_telefonia  = tl.sk_telefonia
""").toPandas()
vis = vis.loc[:, ~vis.columns.str.startswith("sk_")]

ALVO, CHAVE = "flag_churn", "customerID"
X, y, chaves = vis.drop(columns=[ALVO, CHAVE]), vis[ALVO], vis[CHAVE]
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
print(f"{len(vis)} clientes | {len(cat_cols)} categóricas | {len(num_cols)} numéricas")

X_tr, X_te, y_tr, y_te, k_tr, k_te = train_test_split(
    X, y, chaves, test_size=0.25, stratify=y, random_state=SEED)
treino = X_tr.assign(flag_churn=y_tr)
print(f"treino {len(X_tr)} | teste {len(X_te)}")

> **Protocolo.** A divisão treino/teste acontece **antes de qualquer análise**. Regras
> tradicionais, pontos de corte, modelos e limiares são construídos só com o treino.
> Ambas as abordagens são avaliadas no teste, que nenhuma delas viu.
>
> A ressalva registrada na seção 5 vale aqui: sem carimbo de tempo, a divisão é
> aleatória e não temporal. O desempenho reportado é o de reconhecer padrões numa
> população, não o de prever o futuro a partir do passado. É um limite da base, não
> do método.

In [ ]:
# A regra de negócio tradicional: o que um analista monta a partir das tabelas cruzadas.
def regra_tradicional(d):
    p  = (d["Contract"] == "Month-to-month").values.astype(float)
    p += (d["InternetService"] == "Fiber optic").values
    p += (d["PaymentMethod"] == "Electronic check").values
    p += (d["tenure_meses"] < 12).values
    p += (d["TechSupport"] == "No").values
    return p                       # 0 a 5 fatores de risco

pre = ColumnTransformer([("num", StandardScaler(), num_cols),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

modelos = {
  "Regressão Logística": LogisticRegression(max_iter=2000),
  "Random Forest":       RandomForestClassifier(n_estimators=400, min_samples_leaf=5,
                                                random_state=SEED, n_jobs=-1),
  "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                    max_depth=3, random_state=SEED),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

prob_cv, pipes = {}, {}
for nome, m in modelos.items():
    p = Pipeline([("prep", pre), ("m", m)])
    prob_cv[nome] = cross_val_predict(p, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    pipes[nome] = p.fit(X_tr, y_tr)
    print(f"{nome:22s} AUC (validação cruzada no treino) = {roc_auc_score(y_tr, prob_cv[nome]):.4f}")

### 8.2 P2 — Diagnóstica: o que explica o cancelamento

Duas leituras do mesmo fenômeno. A **tradicional** mede a amplitude da taxa de churn
entre categorias de cada variável, uma de cada vez. A do **modelo** usa importância por
permutação: embaralha um atributo e mede quanto o AUC piora, capturando o efeito
*já descontado dos demais*.

In [ ]:
amplitude = {}
for c in cat_cols:
    t = treino.groupby(c)["flag_churn"].agg(["mean", "size"])
    t = t[t["size"] >= 50]                       # ignora categorias raras
    amplitude[c] = t["mean"].max() - t["mean"].min()
for c in num_cols:
    faixas = pd.qcut(treino[c], 4, duplicates="drop")
    t = treino.groupby(faixas, observed=True)["flag_churn"].mean()
    amplitude[c] = t.max() - t.min()
rk_trad = pd.Series(amplitude).sort_values(ascending=False)

MELHOR = max(modelos, key=lambda n: roc_auc_score(y_tr, prob_cv[n]))
pi = permutation_importance(pipes[MELHOR], X_tr, y_tr, scoring="roc_auc",
                            n_repeats=15, random_state=SEED, n_jobs=-1)
rk_mod = pd.Series(pi.importances_mean, index=X_tr.columns).sort_values(ascending=False)

comp = pd.DataFrame({"pos_tradicional": rk_trad.rank(ascending=False).astype(int),
                     "amplitude":       rk_trad.round(4),
                     "pos_modelo":      rk_mod.rank(ascending=False).astype(int),
                     "importancia":     rk_mod.round(5)})
comp["deslocamento"] = comp["pos_tradicional"] - comp["pos_modelo"]
print(f"melhor modelo por AUC: {MELHOR}\n")
display(comp.sort_values("pos_modelo").head(12))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
rk_trad.head(10).iloc[::-1].plot(kind="barh", ax=ax[0], color="#C45A3B")
ax[0].set_title("Tradicional — amplitude da taxa de churn\n(uma variável por vez)")
rk_mod.head(10).iloc[::-1].plot(kind="barh", ax=ax[1], color="#2E6F6B")
ax[1].set_title(f"Modelo — importância por permutação\n({MELHOR}, efeito já descontado das demais)")
plt.tight_layout(); plt.show()

print("Subestimados pela leitura univariada (sobem no modelo):")
display(comp.sort_values("deslocamento", ascending=False).head(4)[["pos_tradicional","pos_modelo","deslocamento"]])
print("Superestimados pela leitura univariada (caem no modelo):")
display(comp.sort_values("deslocamento").head(4)[["pos_tradicional","pos_modelo","deslocamento"]])

print(f"\nAtributo derivado desvio_tarifario (hipótese T8):")
print(f"  posição {int(comp.loc['desvio_tarifario','pos_modelo'])} de {len(comp)} | "
      f"importância {comp.loc['desvio_tarifario','importancia']:.5f}")

**Resposta a P2.** Os dois métodos concordam no essencial: **tempo de casa** e
**modalidade de contrato** lideram nas duas leituras. A convergência é informativa —
não há disputa sobre o que importa mais.

A divergência está no meio da tabela, e é onde a leitura univariada erra:

- **`PaymentMethod` cai da 4ª para a 12ª posição.** Cheque eletrônico tem a maior taxa
  de churn isolada (45,3%), mas quem paga assim é majoritariamente cliente mensal de
  fibra recém-chegado. Controlado por isso, o método de pagamento explica pouco. A
  tabela cruzada estava creditando a ele o efeito do contrato.
- **`valor_mensal` e `ticket_medio_historico` caem 9 posições cada.** Mesmo fenômeno:
  mensalidade alta é consequência de ter fibra, não causa independente de cancelamento.
- **`StreamingMovies`, `StreamingTV` e `MultipleLines` sobem 11 a 13 posições.** Pareciam
  irrelevantes isoladamente e passam a importar quando o modelo as combina com as demais.

**Sobre a hipótese T8 — rejeitada.** O `desvio_tarifario` foi criado para testar se
reajuste recente antecede cancelamento. Ficou em **22º lugar entre 23**, com importância
de 0,0004 — indistinguível de zero. Na base, aumento de tarifa recente não prediz
cancelamento. **Registramos o resultado negativo e mantemos o atributo no catálogo**,
porque a tentativa documentada é informação útil para quem continuar o trabalho.

### 8.3 Segmentação: a comparação precisa de um adversário honesto

Um MVP anterior comparou K-Means contra faixas de tempo de casa e concluiu que a
clusterização separava risco 32% melhor. **Essa comparação é injusta por construção:**
as faixas usam 1 variável, o K-Means usa 19. Refazemos com um terceiro competidor — a
segmentação de negócio com **duas** variáveis, que é o que um analista de fato monta.

Critério: desvio ponderado da taxa de churn entre segmentos (quanto maior, melhor a
separação de risco).

In [ ]:
K = 6
seg_fraco = pd.cut(X_te["tenure_meses"], [-1,12,24,48,100], labels=["0-12","13-24","25-48","49+"])
seg_justo = (X_te["Contract"].replace({"Month-to-month":"Mensal","One year":"1 ano","Two year":"2 anos"})
             + " × " + np.where(X_te["tenure_meses"] < 12, "<1a",
                        np.where(X_te["tenure_meses"] < 36, "1-3a", "3a+")))
km = Pipeline([("prep", pre), ("km", KMeans(n_clusters=K, n_init=20, random_state=SEED))]).fit(X_tr)
seg_km = km.predict(X_te)

def separacao(seg):
    g = pd.DataFrame({"s": np.asarray(seg), "y": y_te.values}).groupby("s")["y"].agg(["mean","size"])
    g = g[g["size"] >= 30]
    w = g["size"] / g["size"].sum()
    return float(np.sqrt((w * (g["mean"] - y_te.mean())**2).sum())), g

res = []
for nome, seg, nv in [("Faixas de tenure", seg_fraco, 1),
                      ("Contrato × tempo de casa", seg_justo, 2),
                      (f"K-Means (k={K})", seg_km, len(X.columns))]:
    sep, g = separacao(seg)
    res.append({"Segmentação": nome, "Variáveis usadas": nv, "Segmentos": len(g),
                "Separação de risco": round(sep, 4),
                "Faixa de churn": f"{g['mean'].min():.1%} – {g['mean'].max():.1%}"})
display(pd.DataFrame(res))
print(f"Silhueta do K-Means: {silhouette_score(km.named_steps['prep'].transform(X_te), seg_km, random_state=SEED):.3f}")
print("\nSegmentação de negócio com 2 variáveis:")
display(separacao(seg_justo)[1].sort_values("mean", ascending=False))

**Resultado: a vantagem da clusterização desaparece.**

| Segmentação | Variáveis | Separação |
|---|---|---|
| Faixas de tenure | 1 | 0,1589 |
| K-Means | 19 | 0,1949 |
| **Contrato × tempo de casa** | **2** | **0,1969** |

Uma tabela cruzada de duas variáveis, montável numa planilha, **supera o K-Means com
19 variáveis**. A silhueta de 0,134 confirma: não existem agrupamentos naturais fortes
nesta base — o K-Means está particionando um espaço contínuo, não descobrindo estrutura.

Isso não desqualifica a clusterização; qualifica a comparação. **O ganho de 32% relatado
anteriormente media a fraqueza do adversário, não a força do método.** O segmento mais
crítico ("Mensal × menos de 1 ano", 52,4% de churn) é encontrado pelas duas abordagens,
mas só a versão de negócio é explicável para o time comercial sem intermediação.

### 8.4 P3 e P4 — Preditiva e calibração

Antes de converter probabilidade em dinheiro, é preciso saber se a probabilidade
significa o que diz. Um modelo pode **ordenar** bem o risco (AUC alto) e ainda assim
estar sistematicamente errado no **nível** (dizer 30% quando o real é 45%). Como todo o
cálculo prescritivo multiplica probabilidades por receita, essa verificação não é
opcional.

Medimos o **escore de Brier** (erro quadrático médio da probabilidade) e o **erro de
calibração esperado (ECE)**: a diferença média entre o previsto e o observado por faixa.

In [ ]:
def ece(y_real, p_prev, n_faixas=10):
    """Erro de calibração esperado: |observado − previsto| ponderado pelo tamanho da faixa."""
    bordas, total = np.linspace(0, 1, n_faixas + 1), 0.0
    for i in range(n_faixas):
        m = (p_prev >= bordas[i]) & (p_prev < bordas[i+1] if i < n_faixas-1 else p_prev <= bordas[i+1])
        if m.sum():
            total += m.sum()/len(p_prev) * abs(y_real[m].mean() - p_prev[m].mean())
    return total

antes = pd.DataFrame([{"modelo": n, "Brier": brier_score_loss(y_tr, prob_cv[n]),
                       "ECE": ece(y_tr.values, prob_cv[n]),
                       "prob. média prevista": prob_cv[n].mean(),
                       "churn real": y_tr.mean()} for n in modelos]).set_index("modelo")
print("Antes da recalibração:"); display(antes.round(4))

# Recalibração isotônica: aprende a curva que corrige o nível das probabilidades
pipes_cal, prob_cv_cal = {}, {}
for nome, m in modelos.items():
    c = CalibratedClassifierCV(Pipeline([("prep", pre), ("m", m)]), method="isotonic", cv=5)
    prob_cv_cal[nome] = cross_val_predict(c, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    pipes_cal[nome] = c.fit(X_tr, y_tr)

depois = pd.DataFrame([{"modelo": n, "Brier": brier_score_loss(y_tr, prob_cv_cal[n]),
                        "ECE": ece(y_tr.values, prob_cv_cal[n])} for n in modelos]).set_index("modelo")
print("Depois da recalibração isotônica:"); display(depois.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for rot, p, cor in [("sem recalibração", prob_cv[MELHOR], "#C45A3B"),
                    ("com recalibração isotônica", prob_cv_cal[MELHOR], "#2E6F6B")]:
    obs, prev = calibration_curve(y_tr, p, n_bins=10, strategy="quantile")
    ax.plot(prev, obs, "o-", label=rot, color=cor)
ax.plot([0,1],[0,1],"k--",lw=1,label="calibração perfeita")
ax.set_xlabel("probabilidade prevista"); ax.set_ylabel("frequência observada de churn")
ax.set_title(f"Curva de calibração — {MELHOR}"); ax.legend(); plt.show()

**Resposta a P4: a suspeita não se confirmou — e isso é um resultado.**

O ECE fica em **1,5%** já sem tratamento: quando o modelo diz 30%, o churn observado é
de 28,5% a 31,5%. A recalibração isotônica reduz para 1,0%, uma melhora marginal.
As probabilidades **são** utilizáveis em cálculo financeiro.

O ponto metodológico permanece: o MVP anterior usou essas probabilidades em contas de
valor esperado **sem nunca ter verificado**. O resultado foi favorável, mas a
verificação é que transforma uma suposição em premissa documentada. Seguimos com os
modelos recalibrados — o custo é nulo e a garantia, explícita.

#### O limiar: F1 ou dinheiro?

Transformar probabilidade em decisão exige um ponto de corte. O MVP anterior escolheu
o limiar que maximiza **F1** — e em seguida converteu o resultado em reais. Há uma
incoerência aí: F1 equilibra precisão e recall com peso igual, o que **não corresponde a
nenhuma realidade econômica**. Aqui, errar um churner custa 12 meses de mensalidade
perdida; errar um cliente fiel custa US$ 30 de contato. Os erros não são simétricos, e
o limiar deveria refletir isso.

Comparamos os dois critérios, cada um escolhido **apenas no treino**.

In [ ]:
CUSTO_CONTATO, TAXA_SUCESSO, MESES = 30.0, 0.30, 12   # premissas ilustrativas

def lucro(y_real, decisao, mensalidade):
    """Receita preservada dos churners abordados − custo de abordar todo mundo."""
    ab = decisao == 1
    return (y_real[ab] * mensalidade[ab]).sum() * TAXA_SUCESSO * MESES - ab.sum() * CUSTO_CONTATO

mensal_tr, grade = X_tr["valor_mensal"].values, np.arange(0.05, 0.95, 0.01)
limiares = {}
for nome in modelos:
    p = prob_cv_cal[nome]
    limiares[nome] = {
      "f1": grade[int(np.argmax([f1_score(y_tr, (p >= t).astype(int)) for t in grade]))],
      "lucro": grade[int(np.argmax([lucro(y_tr.values, (p >= t).astype(int), mensal_tr) for t in grade]))]}
    print(f'{nome:22s} limiar por F1 = {limiares[nome]["f1"]:.2f} | por lucro = {limiares[nome]["lucro"]:.2f}')

pts = regra_tradicional(X_tr)
CORTE_F1    = 1 + int(np.argmax([f1_score(y_tr, (pts >= c).astype(int)) for c in range(1, 6)]))
CORTE_LUCRO = 1 + int(np.argmax([lucro(y_tr.values, (pts >= c).astype(int), mensal_tr) for c in range(1, 6)]))
print(f'{"Regra tradicional":22s} corte por F1 = {CORTE_F1}    | por lucro = {CORTE_LUCRO}')

p = prob_cv_cal[MELHOR]
fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(grade, [f1_score(y_tr, (p>=t).astype(int)) for t in grade], color="#C45A3B", label="F1")
ax1.set_xlabel("limiar"); ax1.set_ylabel("F1", color="#C45A3B")
ax2 = ax1.twinx()
ax2.plot(grade, [lucro(y_tr.values,(p>=t).astype(int),mensal_tr) for t in grade], color="#2E6F6B", label="lucro")
ax2.set_ylabel("lucro esperado (US$)", color="#2E6F6B")
ax1.axvline(limiares[MELHOR]["f1"], ls="--", color="#C45A3B")
ax1.axvline(limiares[MELHOR]["lucro"], ls="--", color="#2E6F6B")
plt.title("Os dois critérios apontam para limiares diferentes"); plt.tight_layout(); plt.show()

In [ ]:
mensal_te = X_te["valor_mensal"].values
pts_te    = regra_tradicional(X_te)
escores   = {"Regra tradicional": pts_te}
dec_f1    = {"Regra tradicional": (pts_te >= CORTE_F1).astype(int)}
dec_lucro = {"Regra tradicional": (pts_te >= CORTE_LUCRO).astype(int)}
for nome in modelos:
    pr = pipes_cal[nome].predict_proba(X_te)[:, 1]
    escores[nome]   = pr
    dec_f1[nome]    = (pr >= limiares[nome]["f1"]).astype(int)
    dec_lucro[nome] = (pr >= limiares[nome]["lucro"]).astype(int)

linhas = []
for nome in escores:
    for rot, d in [("F1", dec_f1[nome]), ("Lucro", dec_lucro[nome])]:
        linhas.append({"Abordagem": nome, "Limiar por": rot,
                       "Acurácia": accuracy_score(y_te, d), "Precisão": precision_score(y_te, d),
                       "Recall": recall_score(y_te, d), "F1": f1_score(y_te, d),
                       "AUC": roc_auc_score(y_te, escores[nome]), "Abordados": int(d.sum()),
                       "Lucro (US$)": lucro(y_te.values, d, mensal_te)})
display(pd.DataFrame(linhas).round(3))

def mcnemar(y_real, a, b):
    ca, cb = (a == y_real), (b == y_real)
    B, C = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return B, C, float(chi2.sf((abs(B-C)-1)**2/(B+C) if B+C else 0.0, 1))
print("Teste de McNemar (limiar por lucro) — regra tradicional × modelo:")
for nome in modelos:
    B, C, pv = mcnemar(y_te.values, dec_lucro["Regra tradicional"], dec_lucro[nome])
    print(f"  {nome:22s} só a regra acertou={B:4d} | só o modelo acertou={C:4d} | p={pv:.5f}")

**Resposta a P3.** Os três modelos superam a regra tradicional, e a diferença é
estatisticamente significativa pelo teste de McNemar (p < 0,0001 em todos). Com o melhor
modelo, o modelo acerta sozinho 210 clientes contra 43 da regra.

**O efeito do critério de limiar é material.** Para a Regressão Logística, o limiar cai de
0,31 (F1) para 0,13 (lucro) e o resultado financeiro sobe de **US$ 79.952 para US$ 85.159
— mais US$ 5.207 com o mesmo modelo e os mesmos dados**, só por decidir com o critério
certo. O F1 cai (0,629 → 0,599), o que é esperado e irrelevante: estamos otimizando
dinheiro, não uma métrica de simetria arbitrária.

Repare na inversão de leitura: pelo limiar de lucro, a abordagem passa a sinalizar 968
clientes em vez de 678, aceitando mais alarmes falsos. É a decisão correta quando o falso
negativo custa 40 vezes mais que o falso positivo.

#### A vantagem sobrevive a outras amostras?

Todo resultado acima vem de **uma** divisão treino/teste. Com 1.761 clientes no teste,
diferenças pequenas podem ser sorte do sorteio. Repetimos o experimento inteiro em **25
divisões independentes**, reconstruindo regra, modelo e limiares em cada uma.

In [ ]:
base_est = Pipeline([("prep", pre), ("m", GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3, random_state=SEED))])
d_auc, d_f1, d_lucro = [], [], []

for s in range(25):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, stratify=y, random_state=1000+s)
    c = CalibratedClassifierCV(base_est, method="isotonic", cv=3).fit(Xa, ya)
    pa_mod, pb_mod = c.predict_proba(Xa)[:,1], c.predict_proba(Xb)[:,1]
    pa_reg, pb_reg = regra_tradicional(Xa), regra_tradicional(Xb)
    ma, mb = Xa["valor_mensal"].values, Xb["valor_mensal"].values
    g = np.arange(0.05, 0.95, 0.01)
    t  = g[int(np.argmax([lucro(ya.values,(pa_mod>=x).astype(int),ma) for x in g]))]
    cc = 1 + int(np.argmax([lucro(ya.values,(pa_reg>=x).astype(int),ma) for x in range(1,6)]))
    dm, dr = (pb_mod >= t).astype(int), (pb_reg >= cc).astype(int)
    d_auc.append(roc_auc_score(yb, pb_mod) - roc_auc_score(yb, pb_reg))
    d_f1.append(f1_score(yb, dm) - f1_score(yb, dr))
    d_lucro.append(lucro(yb.values, dm, mb) - lucro(yb.values, dr, mb))

est = pd.DataFrame({"Δ AUC": d_auc, "Δ F1": d_f1, "Δ Lucro (US$)": d_lucro})
resumo = pd.DataFrame({"média": est.mean(), "desvio": est.std(),
                       "IC95% inferior": est.quantile(.025), "IC95% superior": est.quantile(.975),
                       "% de amostras em que vence": (est > 0).mean()})
display(resumo.round(4))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for i, c_ in enumerate(est.columns):
    ax[i].hist(est[c_], bins=10, color="#2E6F6B", edgecolor="white")
    ax[i].axvline(0, color="#C45A3B", lw=2); ax[i].set_title(c_)
plt.suptitle("Vantagem do modelo sobre a regra em 25 divisões independentes")
plt.tight_layout(); plt.show()

**A superioridade estatística é sólida; a financeira, provável.**

| Métrica | Média | IC 95% | Vence em |
|---|---|---|---|
| Δ AUC | **+0,035** | [+0,022; +0,048] | **100%** das amostras |
| Δ F1 | **+0,058** | [+0,040; +0,074] | **100%** das amostras |
| Δ Lucro | +US$ 2.693 | **[−US$ 952; +US$ 4.368]** | 96% das amostras |

Os dois primeiros intervalos não cruzam zero: a vantagem de discriminação é real e
reprodutível. **O terceiro cruza.** Em 1 de cada 25 trimestres, a campanha guiada por
modelo renderia menos que a guiada pela regra — porque o resultado em dinheiro depende de
*quais* clientes caem na lista e de quanto eles pagam, o que varia mais do que a
capacidade de ordenar risco.

Essa distinção importa para a decisão de operar o modelo. A promessa defensável é "o
modelo ordena risco melhor, de forma consistente". A promessa "o modelo garante mais
lucro todo trimestre" não se sustenta nos dados.

### 8.5 P5 — Prescritiva: qual ação, para quem

Prever quem sai não basta. O gestor precisa de **qual ação oferecer**, dentro de um
orçamento. O cardápio abaixo contém apenas alavancas que a empresa controla e que
apareceram como fatores de risco.

| Ação | Elegibilidade | O que muda | Custo |
|---|---|---|---|
| Contrato anual com desconto | contrato mensal | `Contract → One year` | US$ 60 |
| Migração para débito automático | cheque eletrônico ou boleto | `PaymentMethod → Credit card (automatic)` | US$ 15 |
| Suporte técnico gratuito | tem internet, não tem suporte | `TechSupport → Yes` | US$ 35 |

A lógica é: simular com o modelo o risco **se o cliente aceitasse** cada ação, calcular
`(queda de risco × adesão × receita futura) − custo`, e preencher o orçamento pelos
maiores retornos por dólar.

**Antes disso, porém, uma verificação que faltava.**

#### A simulação "e se?" está pedindo algo que o modelo nunca viu?

Trocar `Contract` de "mensal" para "anual" na planilha é trivial. Mas o modelo aprendeu
com clientes **reais** de contrato anual, que são diferentes dos mensais em muitas outras
dimensões. Se o cliente sintético não se parecer com ninguém da base, a previsão é
**extrapolação**, não estimativa.

Testamos isso com um modelo de propensão: dado tudo **menos** o atributo manipulado (e
seus derivados), quão fácil é distinguir quem está na origem de quem está no destino?
AUC perto de 0,5 significa grupos parecidos e contrafactual confiável. AUC perto de 1
significa que os grupos são estruturalmente distintos e a simulação inventa um cliente.

In [ ]:
ACOES = {
 "Contrato anual":    {"elegivel": lambda d: d["Contract"] == "Month-to-month",
                       "muda": {"Contract": "One year"}, "custo": 60.0},
 "Débito automático": {"elegivel": lambda d: d["PaymentMethod"].isin(["Electronic check","Mailed check"]),
                       "muda": {"PaymentMethod": "Credit card (automatic)",
                                "pagamento_automatico": "Yes"}, "custo": 15.0},
 "Suporte técnico":   {"elegivel": lambda d: (d["TechSupport"] == "No") & (d["InternetService"] != "No"),
                       "muda": {"TechSupport": "Yes"}, "custo": 35.0},
}
# atributos derivados do campo manipulado: saem junto, senão a propensão usa
# o derivado como atalho e a separação fica artificialmente perfeita
DERIVADOS = {"PaymentMethod": ["pagamento_automatico"], "TechSupport": ["n_servicos"], "Contract": []}

suporte = []
for nome, a in ACOES.items():
    col, destino = list(a["muda"].items())[0]
    orig, dest = X_tr[a["elegivel"](X_tr)], X_tr[X_tr[col] == destino]
    comb = pd.concat([orig.assign(_g=0), dest.assign(_g=1)])
    Xc, yc = comb.drop(columns=["_g", col] + DERIVADOS[col]), comb["_g"]
    cc, cn = Xc.select_dtypes(include="object").columns.tolist(), Xc.select_dtypes(exclude="object").columns.tolist()
    prop = Pipeline([("p", ColumnTransformer([("n", StandardScaler(), cn),
                                              ("c", OneHotEncoder(handle_unknown="ignore"), cc)])),
                     ("m", LogisticRegression(max_iter=2000))]).fit(Xc, yc)
    ps = prop.predict_proba(Xc)[:, 1]
    po, pd_ = ps[yc.values == 0], ps[yc.values == 1]
    lo, hi = max(po.min(), pd_.min()), min(po.max(), pd_.max())
    suporte.append({"Ação": nome, "AUC origem × destino": round(roc_auc_score(yc, ps), 3),
                    "Clientes na região de sobreposição": f"{((po>=lo)&(po<=hi)).mean():.1%}",
                    "n origem": len(orig), "n destino": len(dest)})
display(pd.DataFrame(suporte))

**Uma das três ações é extrapolação quase completa.**

| Ação | AUC origem×destino | Sobreposição | Leitura |
|---|---|---|---|
| Débito automático | 0,747 | **98,3%** | confiável |
| Contrato anual | 0,896 | 75,3% | aceitável com ressalva |
| **Suporte técnico** | **0,998** | **19,0%** | **não confiável** |

Quem tem suporte técnico quase sempre tem também segurança online, backup e proteção de
equipamento — são o mesmo tipo de cliente, que contrata o pacote inteiro. Ao ligar
*apenas* o suporte para quem não tem nenhum agregado, a simulação cria um perfil que
praticamente não existe na base: só 19% dos elegíveis têm contraparte comparável.
**O número que sai dali não é estimativa, é extrapolação.**

A ação de débito automático, ao contrário, é segura: quem paga por cheque eletrônico e
quem paga por cartão automático são populações parecidas no resto.

Marcamos cada recomendação com seu nível de confiança no entregável final.

In [ ]:
ORCAMENTO, TAXA_ADESAO = 15000.0, 0.50
modelo_final = pipes_cal[MELHOR]
risco_te     = modelo_final.predict_proba(X_te)[:, 1]
receita_fut  = X_te["valor_mensal"].values * MESES

reducao = pd.DataFrame(index=X_te.index, columns=list(ACOES), dtype=float)
ganho   = pd.DataFrame(index=X_te.index, columns=list(ACOES), dtype=float)
for nome, a in ACOES.items():
    Xm = X_te.copy()
    for c_, v_ in a["muda"].items():
        Xm[c_] = v_
    red = np.where(a["elegivel"](X_te), np.maximum(risco_te - modelo_final.predict_proba(Xm)[:,1], 0), 0.0)
    reducao[nome] = red
    ganho[nome]   = red * TAXA_ADESAO * receita_fut - a["custo"]

def executar(candidatos, orcamento):
    escolhidos, gasto = [], 0.0
    for _, l in candidatos.iterrows():
        if gasto + l["custo"] <= orcamento:
            escolhidos.append(l); gasto += l["custo"]
    return pd.DataFrame(escolhidos), gasto

In [ ]:
pontos_te = regra_tradicional(X_te)

# (A) adversário FRACO — oferta única de contrato anual, por ordem de risco
mA = ACOES["Contrato anual"]["elegivel"](X_te) & (pontos_te >= 2)
candA = pd.DataFrame({"acao": "Contrato anual", "custo": 60.0,
                      "ganho_esperado": ganho.loc[mA, "Contrato anual"],
                      "pontos": pontos_te[mA.values]},
                     index=X_te.index[mA]).sort_values("pontos", ascending=False)

# (B) adversário FORTE — regra de negócio por perfil, como um analista faria
linhasB = []
for i, (idx, row) in enumerate(X_te.iterrows()):
    if pontos_te[i] < 2:
        continue
    if   row["PaymentMethod"] in ("Electronic check", "Mailed check"): acao = "Débito automático"
    elif row["Contract"] == "Month-to-month":                          acao = "Contrato anual"
    elif row["TechSupport"] == "No" and row["InternetService"] != "No":acao = "Suporte técnico"
    else: continue
    linhasB.append({"idx": idx, "acao": acao, "custo": ACOES[acao]["custo"],
                    "ganho_esperado": ganho.loc[idx, acao], "pontos": pontos_te[i]})
candB = pd.DataFrame(linhasB).set_index("idx").sort_values("pontos", ascending=False)

# (C) otimizada por valor esperado
candC = pd.DataFrame({"acao": ganho.idxmax(axis=1), "ganho_esperado": ganho.max(axis=1),
                      "custo": [ACOES[a]["custo"] for a in ganho.idxmax(axis=1)]}, index=X_te.index)
candC = candC[candC["ganho_esperado"] > 0]
candC["retorno_por_dolar"] = candC["ganho_esperado"] / candC["custo"]
candC = candC.sort_values("retorno_por_dolar", ascending=False)

linhas = []
for nome, cand in [("Tradicional — oferta única (adversário fraco)", candA),
                   ("Tradicional — regra por perfil (adversário forte)", candB),
                   ("Otimizada por valor esperado", candC)]:
    p, g = executar(cand, ORCAMENTO)
    linhas.append({"Plano": nome, "Clientes abordados": len(p), "Gasto (US$)": round(g),
                   "Ganho líquido (US$)": round(p["ganho_esperado"].sum()),
                   "ROI": f'{p["ganho_esperado"].sum()/g:.0%}' if g else "—"})
tabela5 = pd.DataFrame(linhas); display(tabela5)

gA, gB, gC = [executar(c, ORCAMENTO)[0]["ganho_esperado"].sum() for c in (candA, candB, candC)]
print(f"ganho da otimização sobre o adversário FRACO : {gC/gA-1:+.1%}")
print(f"ganho da otimização sobre o adversário FORTE : {gC/gB-1:+.1%}")

In [ ]:
sens = []
for orc in [5000, 10000, 15000, 25000, 40000]:
    sens.append({"Orçamento (US$)": orc,
                 "Oferta única": round(executar(candA, orc)[0]["ganho_esperado"].sum()),
                 "Regra por perfil": round(executar(candB, orc)[0]["ganho_esperado"].sum()),
                 "Otimizada": round(executar(candC, orc)[0]["ganho_esperado"].sum())})
sdf = pd.DataFrame(sens).set_index("Orçamento (US$)")
display(sdf)
sdf.plot(marker="o", figsize=(8.5,4.2), color=["#C45A3B","#D9A441","#2E6F6B"])
plt.ylabel("ganho líquido esperado (US$)"); plt.title("Sensibilidade ao orçamento")
plt.grid(alpha=.3); plt.show()

**Resposta a P5: o ganho encolhe pela metade contra um adversário justo — mas sobrevive.**

| Plano | Clientes | Ganho líquido | ROI |
|---|---|---|---|
| Tradicional — oferta única | 250 | US$ 8.851 | 59% |
| Tradicional — regra por perfil | 646 | US$ 13.290 | 89% |
| **Otimizada por valor esperado** | **452** | **US$ 20.318** | **135%** |

Contra a oferta única, a otimização rende **+130%**. Contra a regra por perfil — que
qualquer analista escreveria em dez minutos — rende **+53%**. A diferença entre os dois
números é a medida de quanto uma comparação mal montada infla o resultado.

**Um achado adicional, visível só na curva de sensibilidade:** a regra por perfil
**destrói valor** quando o orçamento cresce. Ela rende US$ 13.300 com US$ 10 mil e apenas
US$ 10.579 com US$ 40 mil. O motivo é que ela ordena por risco, não por retorno, e não
tem critério de parada — continua gastando em clientes cujo ganho esperado é negativo.
O plano otimizado estabiliza em US$ 23.131 porque simplesmente **para** quando o valor
esperado zera. Saber a hora de não agir é parte do valor da abordagem.

### 8.6 P6 — Equidade entre grupos

A base contém atributos demográficos. Antes de operar o modelo, verificamos se ele
distribui atenção de forma equilibrada — tanto por risco reputacional quanto por
conformidade com a Lei 13.709/2018, que exige transparência em decisões automatizadas.

In [ ]:
LIMIAR = limiares[MELHOR]["lucro"]
decisao = (risco_te >= LIMIAR).astype(int)
eq = []
for atr in ["gender", "idoso", "Partner", "Dependents"]:
    for g in sorted(X_te[atr].unique()):
        m = (X_te[atr] == g).values
        eq.append({"atributo": atr, "grupo": g, "n": int(m.sum()),
                   "churn real": y_te.values[m].mean(), "taxa de seleção": decisao[m].mean(),
                   "recall": recall_score(y_te.values[m], decisao[m]),
                   "precisão": precision_score(y_te.values[m], decisao[m], zero_division=0),
                   "AUC": roc_auc_score(y_te.values[m], risco_te[m])})
eqd = pd.DataFrame(eq); display(eqd.round(3))
print("Disparidade máxima dentro de cada atributo:")
for a in eqd["atributo"].unique():
    s = eqd[eqd.atributo == a]
    print(f"  {a:12s} Δrecall = {s['recall'].max()-s['recall'].min():.3f} | "
          f"AUC de {s['AUC'].min():.3f} a {s['AUC'].max():.3f}")

**Resposta a P6: há disparidade, e ela não é do tipo que se esperava.**

- **Gênero: nenhum efeito.** Δrecall de 0,009. O atributo não carrega sinal e poderia ser
  removido do modelo sem perda.
- **Idosos: selecionados em excesso e classificados pior.** 80,4% dos clientes idosos são
  sinalizados, contra 50,0% dos demais. Parte disso é legítimo — o churn real deles é de
  42,7% contra 23,4%. Mas o **AUC cai para 0,798** contra 0,847: o modelo **ordena pior**
  justamente o grupo que mais aborda. Com 286 idosos no teste, há pouco dado para
  aprender o padrão, e a abordagem em massa compensa a imprecisão com volume.
- **Dependentes: a maior disparidade de recall (0,143).** O modelo captura 94,7% dos
  churners sem dependentes e apenas 80,5% dos com dependentes. Um em cada cinco clientes
  com dependentes que vai cancelar passa despercebido.

Nenhuma disparidade é grave a ponto de impedir o uso, mas duas exigem registro: o modelo
é menos confiável para idosos e menos sensível para clientes com dependentes. Em operação,
isso pediria limiares por grupo ou coleta adicional nesses segmentos.

---
## 9. Entregável — plano de ação persistido

O produto final do pipeline: lista priorizada, **com o identificador real do cliente**,
a ação recomendada, o ganho esperado e o nível de confiança do contrafactual.

> Uma versão anterior deste trabalho descartava o `customerID` na limpeza e exportava o
> plano indexado pela posição na tabela. O arquivo era inutilizável: o time comercial não
> tinha como localizar os clientes. A chave natural agora atravessa todo o pipeline
> preservada em `dim_cliente`.

In [ ]:
CONFIANCA = {"Débito automático": "alta", "Contrato anual": "média", "Suporte técnico": "baixa — extrapolação"}

plano = executar(candC, ORCAMENTO)[0].copy()
plano["customerID"]     = k_te.loc[plano.index].values
plano["risco_atual"]    = risco_te[[X_te.index.get_loc(i) for i in plano.index]]
plano["reducao_risco"]  = [reducao.loc[i, a] for i, a in zip(plano.index, plano["acao"])]
plano["confianca_simulacao"] = plano["acao"].map(CONFIANCA)
plano = plano.join(X_te[["tenure_meses","Contract","PaymentMethod","valor_mensal"]])
plano["prioridade"] = range(1, len(plano)+1)
plano = plano[["prioridade","customerID","risco_atual","acao","confianca_simulacao",
               "reducao_risco","custo","ganho_esperado","tenure_meses","Contract",
               "PaymentMethod","valor_mensal"]]

# persiste nos dois formatos: tabela Delta (consultável) e CSV (operacional)
(spark.createDataFrame(plano).write.mode("overwrite").option("overwriteSchema", True)
      .saveAsTable(FQ("gold_plano_retencao")))
plano.to_csv(f"{VOLUME}/plano_de_acao_retencao.csv", index=False, float_format="%.4f")

print(f"{len(plano)} clientes no plano")
print(f"tabela Delta : {FQ('gold_plano_retencao')}")
print(f"arquivo CSV  : {VOLUME}/plano_de_acao_retencao.csv\n")
print("Mix de ações e confiança da simulação:")
display(plano.groupby(["acao","confianca_simulacao"]).agg(
    clientes=("customerID","size"), ganho_total=("ganho_esperado","sum")).round(0))
display(plano.head(10))

In [ ]:
%sql
-- Prova de persistência: as tabelas existem fora da sessão
SHOW TABLES IN workspace.churn_mvp

---
## 10. Discussão geral

As seis perguntas, respondidas em conjunto, descrevem um sistema de retenção viável
com ressalvas conhecidas.

**O churn é um problema de receita, não de contagem** (P1). Perde-se 30,5% do
faturamento com 26,5% dos clientes porque quem sai paga 21% a mais. Isso eleva o teto do
que faz sentido gastar em retenção e explica por que o ROI do plano otimizado chega a
135%.

**A leitura univariada erra de forma sistemática, e erra para cima** (P2). Forma de
pagamento e mensalidade parecem causas e são consequências de contrato e tecnologia.
Uma campanha desenhada pela tabela cruzada investiria em mudar o método de pagamento
acreditando num efeito que, controlado, é pequeno. O modelo não substitui a leitura de
negócio — foi ela que gerou as hipóteses —, mas corrige a atribuição de crédito.

**A vantagem do modelo é real na ordenação e incerta no dinheiro** (P3). Δ AUC de +0,035
vencendo em 100% das amostras é um resultado forte. Δ Lucro com intervalo cruzando zero
é um resultado honesto. Operar o modelo é justificável; prometer ganho financeiro
garantido, não.

**A verificação de calibração mudou o status da premissa, não o número** (P4). O ECE de
1,5% autoriza o uso das probabilidades em conta financeira. A diferença entre este
trabalho e o anterior não é o resultado, é saber que ele vale.

**O ganho prescritivo encolhe de 130% para 53% quando o adversário é justo** (P5). Esse é
o achado metodológico central. Comparações contra versões deliberadamente fracas da
abordagem tradicional produzem vantagens infladas. E a descoberta de que a regra de
negócio destrói valor com orçamento alto, por não saber parar, é mais acionável do que a
diferença de métrica.

**A simulação prescritiva não tem suporte uniforme** (P5/P6). Das três ações, uma é
confiável, uma é aceitável e uma é extrapolação. Sem a checagem de sobreposição, as três
sairiam no relatório com a mesma aparência de precisão.

**Dois grupos recebem tratamento desigual** (P6). Idosos são abordados em excesso com
modelo menos confiável; clientes com dependentes são sub-detectados em 14 pontos de
recall.

**A limitação que atravessa tudo**: a ausência de carimbo de tempo impede validação
temporal. O que medimos é capacidade de reconhecer padrões numa população, não de prever
o futuro. Nenhum refinamento de modelo resolve isso — só dado melhor resolve.

---

## 11. Autoavaliação

**Quais perguntas foram respondidas e quais não?**

| | Pergunta | Situação |
|---|---|---|
| P1 | Receita em risco | **Respondida.** US$ 139.131/mês, 30,5% do faturamento, concentrada em mensal + fibra + cheque eletrônico |
| P2 | Fatores explicativos | **Respondida.** Contrato e tempo de casa lideram; pagamento e mensalidade são consequência, não causa. A hipótese T8 foi testada e **rejeitada** |
| P3 | Previsibilidade | **Respondida com ressalva.** Ganho de AUC sólido (100% das amostras); ganho financeiro não garantido (IC cruza zero) |
| P4 | Confiabilidade das probabilidades | **Respondida.** ECE de 1,5%; probabilidades utilizáveis em cálculo financeiro |
| P5 | Melhor alocação do orçamento | **Respondida parcialmente.** O plano é mensurável, mas 17 das 452 recomendações apoiam-se em contrafactual sem suporte |
| P6 | Equidade | **Respondida.** Sem viés de gênero; disparidade relevante para idosos e clientes com dependentes |

Nenhuma pergunta ficou sem resposta, mas **P5 só é respondida de forma condicional**: o
valor esperado é uma estimativa de associação, não de causa. A resposta definitiva exige
experimento, não dado observacional.

**Que limitações dos dados condicionaram os resultados?**

1. **Ausência de marcação temporal** — a mais séria. Impede validação temporal, cálculo de janela de observação e análise de sobrevivência, que é o método correto para churn.
2. **Snapshot único** — sem histórico, não há trajetória: só o estado final do cliente.
3. **Ausência de dados comportamentais** — chamados abertos, reclamações, qualidade do sinal, uso de dados. São justamente os sinais que antecedem o cancelamento, e nenhum está na base.
4. **Base de um único setor e uma única operadora** — nada autoriza generalizar os coeficientes.
5. **Sobreposição limitada para contrafactuais** — restringe a confiabilidade da camada prescritiva, como medido na seção 8.5.
6. **Premissas financeiras arbitradas** — custo de contato, taxa de sucesso e horizonte são ilustrativos. Comparam abordagens entre si; não são previsão de receita.

**Que decisões técnicas eu tomaria de outra forma?**

1. **Começaria pela modelagem dimensional, não pelo modelo preditivo.** Na primeira versão, a análise veio antes da estrutura, e o `customerID` foi descartado na limpeza — o que inutilizou o entregável final. A chave natural é requisito de produto, não detalhe de limpeza.
2. **Teria escolhido o limiar por valor esperado desde o início.** Otimizar F1 e depois converter em dinheiro é incoerente, e custou US$ 5.207 no mesmo modelo.
3. **Teria construído o adversário tradicional na sua melhor versão.** Comparar contra uma versão fraca produziu uma conclusão ("a IA vence em 5 de 5") que não sobreviveu ao teste justo na segmentação.
4. **Teria verificado o suporte do contrafactual antes de publicar números prescritivos**, não depois.
5. **Usaria `class_weight` com mais cuidado.** Reponderar classes distorce a probabilidade e teria atrapalhado a calibração, que aqui é pré-requisito do cálculo financeiro.

**Que extensões transformariam este MVP em solução de uso contínuo?**

| Prioridade | Extensão | Por quê |
|---|---|---|
| 1 | **Teste A/B das ações** (tratado × controle) | única forma de transformar associação em efeito causal e validar os US$ 20.318 |
| 2 | **Ingestão incremental com carimbo de tempo** (Auto Loader + Delta MERGE) | habilita validação temporal e dimensão de histórico (SCD tipo 2) |
| 3 | **Modelos de *uplift*** | estimam o efeito incremental por cliente, em vez da diferença de risco simulada |
| 4 | **Monitoramento de deriva** sobre `qa_qualidade_atributos` | o perfil de qualidade já é tabela; falta compará-lo entre cargas e disparar alerta |
| 5 | **Explicação individual (SHAP)** no plano de ação | o atendente precisa saber *por que* aquele cliente está na lista |
| 6 | **Orquestração em Databricks Workflows** | o pipeline roda hoje por execução manual; produção exige agendamento, retentativa e alerta de falha |
| 7 | **Limiares por segmento** | corrige a disparidade de recall medida em P6 |

---

## 12. Ética, licenciamento e proteção de dados

**Licença.** Database Contents License (DbCL) v1.0, que autoriza uso, modificação e
redistribuição mediante atribuição. O repositório versiona o código e o notebook; o CSV
bruto não é redistribuído — o caminho de obtenção está documentado na seção 3.

**Dados pessoais.** A base não contém identificadores diretos. `customerID` é código
sintético sem vínculo com pessoa natural, e nenhum atributo permite reidentificação.
Não se aplica tratamento adicional sob a Lei 13.709/2018.

**Decisão automatizada.** Ainda que anonimizada, a aplicação ilustra uma decisão
automatizada que afeta clientes: quem recebe oferta e qual. O art. 20 da LGPD assegura
revisão. Por isso o plano exportado carrega risco, ação, redução estimada **e o nível de
confiança da simulação** — e por isso a seção 8.6 audita disparidade entre grupos. Um
sistema que recomenda sem expor sua base de decisão não é auditável.

**Atributos sensíveis.** Gênero entra no modelo e demonstra-se sem efeito (Δrecall
0,009). Em operação real, a recomendação é **removê-lo**: um atributo sem poder
preditivo e com potencial de discriminação só adiciona risco.